# 02 — Training (D3–D6)

**YOLOv8s / YOLO11s / YOLO26s under identical settings.**

The paper's entire claim is *identical conditions*. Everything comes from `src/config.py`. **Do not tune per model** — a different lr for one model kills the controlled comparison and the contribution with it.

**Gate D4:** YOLOv8s mAP@0.4 < 0.20 on positive-only CV → labels or fusion are broken. Back to notebook 01. Don't launch two more models on bad data.

---
### ⚠️ Accelerator: `GPU T4 x2` — NOT P100

P100 is **sm_60**; current PyTorch requires **sm_70+**. Verified on a live session:

```
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the
current PyTorch installation. Supports: sm_70 sm_75 sm_80 sm_86 sm_90 ...
```

Training fails or silently falls back to CPU. Select **`GPU T4 x2`** (sm_75), keep `device=0`.

| | T4 | P100 |
|---|---|---|
| Compute capability | **sm_75 — supported** | sm_60 — **unusable** |
| Tensor cores | **Yes** (real FP16 speedup) | No |
| Bandwidth | 320 GB/s | 732 GB/s |

P100 wins on bandwidth, which is why it looked better on paper. Irrelevant — it doesn't run.

**Don't attempt 2×T4 DDP.** Kaggle bills by wall-clock not per-GPU, so `T4 x2` with `device=0` costs the same quota. Notebook DDP spawn failures are a known sink — best case ~2 h saved, worst case a lost day.

T4 has FP16 tensor cores but **no bf16**. `amp=True`, never `bf16`.

---
### Prerequisite — attach notebook 01's output

1. In **notebook 01**: Save Version → **"Save & Run All (Commit)"**, *not* Quick Save. Only a full commit persists `/kaggle/working` as notebook output. Takes ~10 min. Push to GitHub first — the commit re-clones.
2. Here: **Add Data** (top right) → **"Notebook Output Files"** tab → filter **"Your Work"** → select notebook 01.

`C.find_data_yaml()` locates it from wherever it mounts. If it raises, the attach hasn't happened or the commit is still running.

In [ ]:
!pip install -q ultralytics wandb

REPO = "/kaggle/working/repo"
!rm -rf {REPO} && git clone -q https://github.com/ShMazumder/vinbigdata-chest-x-ray.git {REPO}

import sys
sys.path.insert(0, REPO)

from pathlib import Path
import pandas as pd

from src import config as C
from src.training import train as T
from src.utils.run_logger import RunLogger, benchmark_inference, capture_gpu

!cd {REPO} && git rev-parse --short HEAD

In [ ]:
# strict=True -> raises immediately on sm_60 (P100). Fails in seconds rather
# than after a session of confusing CPU-speed epochs.
gpu = capture_gpu(strict=True)
gpu

In [ ]:
# Locates data.yaml wherever the notebook output mounted, and hard-fails if the
# images are dangling links -- the failure that would otherwise surface two
# hours into training.
DATA_YAML = C.find_data_yaml()
DATA_ROOT = DATA_YAML.parent
print(DATA_YAML.read_text())

## 1. W&B overhead measurement (D3, ~10 min)

5 epochs with and without W&B, compare `sec_per_epoch` from `RunLogger`.

General figure is 1–3% (async background process; **image logging is the expensive part, not scalars**). Measure it on *your* hardware rather than trusting that. If >5%, disable image logging and keep the scalars.

This also validates the whole training path end-to-end before committing 2 hours to it — and gives you the real per-epoch time on T4, which the compute budget depends on.

In [ ]:
_ = T.train_one("yolov8s", DATA_YAML, epochs=5, use_wandb=False,
                runs_dir=C.RUNS_DIR / "smoke_nowandb")

In [ ]:
# Needs WANDB_API_KEY in Add-ons -> Secrets.
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    have_wandb = True
except Exception as e:
    print(f"no W&B secret ({e}) -- continuing without. RunLogger still records everything.")
    have_wandb = False

if have_wandb:
    _ = T.train_one("yolov8s", DATA_YAML, epochs=5, use_wandb=True,
                    runs_dir=C.RUNS_DIR / "smoke_wandb")

In [ ]:
a = RunLogger.master_table(C.RUNS_DIR / "smoke_nowandb").sec_per_epoch.iloc[-1]
print(f"no-wandb: {a:.1f}s/epoch -> {a*40/60:.0f} min per model, {a*40*3/3600:.1f} h for all three")

if have_wandb:
    b = RunLogger.master_table(C.RUNS_DIR / "smoke_wandb").sec_per_epoch.iloc[-1]
    print(f"wandb:    {b:.1f}s/epoch -> {b*40/60:.0f} min per model  (overhead {(b/a-1)*100:.1f}%)")

## 2. Full training runs (D3–D5)

Sequential, one GPU. **Checkpoints save every epoch** — the 12 h session cap will interrupt you at some point; re-run with `resume=True` on the affected model.

If §1 projects more than ~3 h total, reconsider before starting: reduce to 2 models (YOLOv8s vs YOLO26s — the NMS vs NMS-free contrast, which is Axis B) rather than cutting epochs. Protocol consistency matters more than model count.

In [ ]:
results = T.train_all(DATA_YAML, use_wandb=have_wandb)
{k: v["status"] for k, v in results.items()}

In [ ]:
# Session killed mid-run? Resume just that model:
# _ = T.train_one("yolo26s", DATA_YAML, resume=True)

## 3. mAP@0.4 — the competition metric

> ⚠️ **Ultralytics cannot give you mAP@0.4.** It reports @0.5 and @0.5:0.95 only. VinBigData used IoU > 0.4.
>
> If @0.5 silently lands in a column labelled @0.4, the entire related-work comparison misaligns — and that error survives to camera-ready.

**Report both:** @0.4 for the competition lineage (0.253 / 0.314), @0.5 for modern literature (0.338 / 0.415 / 0.453).

In [ ]:
from ultralytics import YOLO
from src.evaluation import metrics as M

weights = T.all_best_weights()
evals = {}
for key, w in weights.items():
    print(f"\n=== {key} ===")
    evals[key] = M.evaluate_full(
        YOLO(w), DATA_ROOT / "images" / "test", DATA_ROOT / "labels" / "test",
        imgsz=C.IMGSZ,
    )

In [ ]:
summary = pd.DataFrame({
    k: {"mAP@0.4": v["map40"], "mAP@0.5": v["map50"]} for k, v in evals.items()
}).T.round(4)
print(summary)

print("\nPublished bar — context only, NOT a target:")
for name, vals in C.PUBLISHED_BASELINES.items():
    print(f"  {name}: {vals}")

### ⛔ Gate D4

Reference reached ~0.45 CV positive-only with EfficientDet at 896–1024px, 50 epochs. You're at 512px, `s`-scale, 40 epochs — less model, less resolution.

- **0.25–0.35** — reasonable, proceed
- **< 0.20** — labels or fusion are broken, not the model. Stop.
- **> 0.45** — suspicious. Check the split for leakage before believing it.

In [ ]:
m = evals["yolov8s"]["map40"]
if m < 0.20:
    print(f"⛔ GATE FAILED: mAP@0.4={m:.4f} < 0.20. Debug the data pipeline.")
elif m > 0.45:
    print(f"⚠ mAP@0.4={m:.4f} above the reference's 896px EfficientDet. Check for leakage.")
else:
    print(f"✓ gate passed: mAP@0.4={m:.4f}")

In [ ]:
# Per-class with n, sorted ascending. Reporting rule was fixed in notebook 01
# BEFORE any score was seen: keep all 14 in mAP, show n, caveat n<30 in
# limitations. Pneumothorax n=10, Atelectasis n=22.
pc = pd.DataFrame({k: v["detail_40"]["per_class"] for k, v in evals.items()}).round(4)
pc["n_gt"] = pd.Series(list(evals.values())[0]["detail_40"]["n_gt"])
pc.sort_values("n_gt")

## 4. Inference benchmark (D10) — same card, one session

> ⚠️ **mAP is hardware-independent. Latency is not.**
>
> YOLO26 is marketed as edge-first (claimed 43% CPU speed gain over YOLO11), so a latency table is expected. It is only valid if produced **here** — all three checkpoints back to back, one session, one card — not stitched from training runs that may have landed on different GPUs.

In [ ]:
RunLogger.check_hardware_consistency(C.RUNS_DIR)
bench = benchmark_inference(weights, imgsz=C.IMGSZ, out_dir=str(C.RUNS_DIR))
pd.DataFrame(bench).T[["ms_per_image", "fps", "n_params_M", "gpu_family"]]

## 5. Persist

`/kaggle/working` dies with the session. Commit via **Save Version → "Save & Run All"** so notebook 03 can attach these weights.

In [ ]:
import json
(C.RUNS_DIR / "eval_summary.json").write_text(json.dumps(
    {k: {"map40": v["map40"], "map50": v["map50"],
         "per_class_40": v["detail_40"]["per_class"],
         "n_gt": v["detail_40"]["n_gt"]} for k, v in evals.items()},
    indent=2, default=str))

print("weights:")
for k, w in weights.items():
    print(f"  {k}: {w}")
print("\nNext: Save Version -> 'Save & Run All (Commit)', then attach to notebook 03.")